In the bigram transformer, we tokenized by creating a map, and mapping each character in the shakespeare text to it's value. This was character level tokenization, which works, but it naive and weak. LLMs like ChatGPT use chunk-level tokenization, which is done in many different ways

Strings are just sequences of unicode codepoints. Unicode is the universal standard that assigns a unique number to every character (not just english, other languages, numbers, emojis, etc). To access unicode, we can use the ord() function.

In [2]:
ord('h'), ord('🙃')

(104, 128579)

In [4]:
[ord(x) for x in 'hello 😃']

[104, 101, 108, 108, 111, 32, 128515]

We can't use unicode as a native tokenizer, as it represents ≈150k characters... A dictionary too long. It also keeps getting updated. Therefore, we need something better. Lets look at encodings.

Encodings (for eg. UTF-8, UTF-16, etc) translate unicode text into byte-wise data (each byte holds a value 255 or under).

In [7]:
'hello 😃'.encode("utf-8")

b'hello \xf0\x9f\x98\x83'

In [6]:
list('hello 😃'.encode("utf-8"))
# The last 4 bytes accounts for the emoji

[104, 101, 108, 108, 111, 32, 240, 159, 152, 131]

The problem with just using an encoding is that it would result in encoded input strings being extremely long (for eg, every space would take up a byte, or every emoji would take up 4 bytes like the one seen above, etc). As a result there may be some attention issues with longer input strings.

 A solution to this the Byte-Pair Encoding (BPE) algorithm which will allow us to compress these sequences.

Starting with a string (below). This is an 11 character string with a vocabulary size of 4.
- aaabdaaabac

First, find the most commonly occurring pair of values. In this case, it would be aa. Replace aa with a new value, Z (which we append to our vocabulary). The new string (below), is 9 characters long, with a larger vocabulary of 5.
- ZabdZabac
- Z = aa

Do it again. This time the pair is ab. Let Y = ab. The result string is 7 characters long with a vocabulary size of 6.
- ZYdZYac
- Z = aa
- Y = ab

Do it again. This time its ZY, which we can set to X. The next string is 5 characters long with a vocabulary size of 7.
- XdXac
- Z = aa
- Y = ab
- X = ZY

Dummy implementation below:

In [10]:
text = "Unicode! UNICODE? [U][N][I][C][O][D][E]! 😂 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to 'support Unicode' in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don't blame programmers for still finding the whole thing mysterious, even 30 years after Unicode's inception."
tokens = text.encode("utf-8") # Raw bytes
tokens = list(map(int, tokens)) # Convert the bytes to a list of integers
print('---')
print(text)
print(len(text))
print('---')
print(tokens)
print(len(tokens)) # Should probably be longer, as ASCII characters take up a byte, but special character, or emojis can take up to 4

---
Unicode! UNICODE? [U][N][I][C][O][D][E]! 😂 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to 'support Unicode' in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don't blame programmers for still finding the whole thing mysterious, even 30 years after Unicode's inception.
541
---
[85, 110, 105, 99, 111, 100, 101, 33, 32, 85, 78, 73, 67, 79, 68, 69, 63, 32, 91, 85, 93, 91, 78, 93, 91, 73, 93, 91, 67, 93, 91, 79, 93, 91, 68, 93, 91, 69, 93, 33, 32, 240, 159, 152, 130, 32, 84, 104, 101, 32, 118, 101, 114, 121, 32, 110, 97, 109, 101, 32, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 101, 32, 105, 110, 116, 111, 32, 116, 104, 101, 32, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32,

In [13]:
# First, lets find the most commonly occuring pair of bytes
def get_most_common_pair(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

most_common_pair = get_most_common_pair(tokens)
# print(most_common_pair)
print(sorted(((v,k) for k, v in most_common_pair.items()), reverse=True))

[(20, (101, 32)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (97, 114)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (100, 101)), (6, (93, 91)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (111, 100)), (5, (110, 105)), (5, (110, 32)), (5, (105, 99)), (5, (99, 111)), (5, (85, 110)), (5, (44, 32)), (5, (32, 115)), (5, (32, 85)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (110, 116)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (97, 109)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (3, (108, 101)), (3, (108, 32)), (3, (101, 115)), (3, (101, 110)), (3, (97, 116)), (3, (46, 32)), (3, (32, 

The most common pair right now is 101, 32, which occurs 20 times. It is an e followed by a space.

In [15]:
chr(101), chr(32)

('e', ' ')

Lets mint a new token, 256 which we can then use to replace every occurance of (101, 32).

In [21]:
top_pair = max(most_common_pair, key=most_common_pair.get)
print(top_pair)

(101, 32)


In [22]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# Print the result of merging the orders pari 6, 7 with 99 in the string [5, 6, 6, 7, 9, 1]
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print(len(tokens2)) # 546 down to 526 as expected since the pair occurred 20x

[5, 6, 99, 9, 1]
[85, 110, 105, 99, 111, 100, 101, 33, 32, 85, 78, 73, 67, 79, 68, 69, 63, 32, 91, 85, 93, 91, 78, 93, 91, 73, 93, 91, 67, 93, 91, 79, 93, 91, 68, 93, 91, 69, 93, 33, 32, 240, 159, 152, 130, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 32, 39, 115, 117, 112, 112, 111, 114, 116, 32, 85, 110, 105, 99, 111, 100, 101, 39, 32, 105, 110, 32, 111, 117, 114, 32, 115, 111, 102, 116, 119, 97, 114, 256, 40, 119, 104, 97, 116, 101, 118, 101, 114, 32, 116, 104, 97, 116, 32, 109, 101, 97, 110, 115, 226, 128, 148, 108, 105, 107, 256, 117, 115, 105, 110, 103, 32, 119, 99, 104, 97, 114, 

Now lets write a loop that repeatedly does this.

In [23]:
# First, lets find the most commonly occuring pair of bytes
def get_most_common_pair(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

most_common_pair = get_most_common_pair(tokens)
print(sorted(((v,k) for k, v in most_common_pair.items()), reverse=True))


def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


vocab_size = 276 # (hyperparameter) --> the desired final vocabulary size
num_merges = vocab_size - 256 # 20 in this case
ids = list(tokens) # making a copy
merges = {} # (int, int) -> int
for i in range(num_merges):
    stats = get_most_common_pair(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

[(20, (101, 32)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (97, 114)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (100, 101)), (6, (93, 91)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (111, 100)), (5, (110, 105)), (5, (110, 32)), (5, (105, 99)), (5, (99, 111)), (5, (85, 110)), (5, (44, 32)), (5, (32, 115)), (5, (32, 85)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (110, 116)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (97, 109)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (3, (108, 101)), (3, (108, 32)), (3, (101, 115)), (3, (101, 110)), (3, (97, 116)), (3, (46, 32)), (3, (32, 

In [26]:
print("tokens length: ", len(tokens))
print("ids length: ", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}")

tokens length:  546
ids length:  404
compression ratio: 1.35


The tokenizer is essentially the middleman between the actual model and the input string. The model does not see a single character/letter... Only numbers (tokens).

Now that we've gone through compression of a string into byte-level values, The next step is to learn the general process to decode and encode any string into tokens, or token sequence back into a string.

In [ ]:
vocab = {idx: bytes[idx] for idx in range(256)}
for (p0, p1) in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    #  Given a list of integers, return a string
    tokens = "".join(vocab[idx] for idx in ids)
    text = tokens.decod("utf-8", errors="replace") # By default, errors = strict
    # Must define errors as starting bytes may not be valid for UTF8 encoding
    return text